# Data Collection — Terna API

This notebook collects public data on renewable electricity production in Italy from the Terna API.

The data will be used to analyze:
- temporal trends in renewable electricity production;
- regional differences;
- the contribution of different renewable energy sources.

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

## Research Question

How has renewable electricity production in Italy changed over time, and how does it differ across Italian regions?

The analysis will focus on:
- the evolution of renewable electricity production over time;
- differences between Italian regions;
- the contribution of different renewable sources, such as hydro, solar, wind, geothermal, and bioenergy.

## Required Data

The analysis requires data with the following dimensions:

- **Time**: year
- **Geography**: Italian region and province
- **Renewable source**: hydro, solar, wind, geothermal, bioenergy
- **Measure**: gross electricity production (GWh)

## Time Period

The analysis covers the period from **2000 to 2024**, providing 25 years of annual data on renewable electricity production in Italy.

## Terna API Endpoint

Renewable electricity production data will be collected from the Terna Renewable Sources Production API.

**Endpoint:** `GET /generation/v2.0/renewable-sources-production`

**Required parameter:**
- `year`

**Optional filters:**
- `region`
- `province`
- `source`

**Renewable sources:**
- Bioenergie
- Eolico
- Fotovoltaico
- Geotermoelettrico
- Idrico

**Main response fields:**
- `year`
- `production_type`
- `region`
- `province`
- `renewable_source`
- `production_GWh`

## Authentication

The Terna API uses OAuth 2.0 with the Client Credentials flow.

API credentials will be stored locally in a `.env` file and will not be committed to the Git repository.

The authentication process will be:

1. Load the API credentials from `.env`.
2. Request an access token from Terna.
3. Use the access token to authenticate API requests.
4. Collect the renewable electricity production data.

## Expected Dataset Structure

The cleaned dataset is expected to contain the following columns:

| Column | Description |
|---|---|
| `year` | Reference year |
| `region` | Italian region |
| `province` | Italian province |
| `renewable_source` | Renewable energy source |
| `production_GWh` | Gross electricity production in GWh |

## Data Quality Checks

After collecting the data, the following checks will be performed:

- inspect the dataset shape;
- check column names and data types;
- identify missing values;
- check for duplicate records;
- verify the available years;
- verify the Italian regions;
- verify the renewable energy sources.

In [2]:


RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(RAW_DATA_DIR.resolve())
print(PROCESSED_DATA_DIR.resolve())

/Users/gianpietrocolato/Desktop/renewable-energy-italy-analysis/data/raw
/Users/gianpietrocolato/Desktop/renewable-energy-italy-analysis/data/processed


## Preliminary Excel Inspection

An initial Terna Excel file for 2000 is inspected to understand the available variables and data structure. This file is used only for preliminary exploration and is not used as the analytical data source.

The final dataset is collected directly from the Terna API for the complete 2000–2024 period.

In [3]:
file_2000 = RAW_DATA_DIR / "terna_electricity_production_2000.xlsx"
df_2000 = pd.read_excel(file_2000)

/Users/gianpietrocolato/miniconda3/envs/data_env/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
df_2000.head()

,Anno,Tipo produzione,Regione,Provincia,Fonte,Produzione (GWh)
0,2000,Lorda,Abruzzo,Chieti,Eolico,0.186600
1,2000,Netta,Campania,Salerno,Fotovoltaico,4.111070
2,2000,Netta,Campania,Salerno,Idrico,188.860704
3,2000,Netta,Campania,Salerno,Termoelettrico,23.390064
4,2000,Lorda,Emilia-Romagna,Bologna,Eolico,2.563731


In [5]:
df_2000.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Anno              417 non-null    object 
 1   Tipo produzione   416 non-null    object 
 2   Regione           416 non-null    object 
 3   Provincia         416 non-null    object 
 4   Fonte             416 non-null    object 
 5   Produzione (GWh)  416 non-null    float64
dtypes: float64(1), object(5)
memory usage: 19.7+ KB


In [6]:
df_2000[df_2000.isna().any(axis=1)]

,Anno,Tipo produzione,Regione,Provincia,Fonte,Produzione (GWh)
416,Applied filters: Year is 2000; Month is gennaio;,NaN,NaN,NaN,NaN,NaN


In [7]:
# Remove the metadata row and keep only valid production records.
df_2000_clean = df_2000.dropna(subset=["Tipo produzione"]).copy()

In [8]:
df_2000_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 416 entries, 0 to 415
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Anno              416 non-null    object 
 1   Tipo produzione   416 non-null    object 
 2   Regione           416 non-null    object 
 3   Provincia         416 non-null    object 
 4   Fonte             416 non-null    object 
 5   Produzione (GWh)  416 non-null    float64
dtypes: float64(1), object(5)
memory usage: 22.8+ KB


In [9]:
# Convert the year column from object to integer.
df_2000_clean["Anno"] = df_2000_clean["Anno"].astype(int)

In [10]:
df_2000_clean.dtypes

Anno                  int64
Tipo produzione      object
Regione              object
Provincia            object
Fonte                object
Produzione (GWh)    float64
dtype: object

In [11]:
df_2000_clean["Fonte"].value_counts()

Fonte
Termoelettrico       194
Idrico               166
Eolico                32
Fotovoltaico          16
Geotermoelettrico      8
Name: count, dtype: int64

In [12]:
df_2000_clean["Tipo produzione"].value_counts()

Tipo produzione
Lorda    208
Netta    208
Name: count, dtype: int64

In [13]:
# Check whether each region-province-source combination contains both production types.
df_2000_clean.groupby(
    ["Regione", "Provincia", "Fonte"]
)["Tipo produzione"].nunique().value_counts()

Tipo produzione
2    208
Name: count, dtype: int64

In [14]:
# Inspect the first records after sorting by geographical and production dimensions.
df_2000_clean.sort_values(
    ["Regione", "Provincia", "Fonte", "Tipo produzione"]
).head(10)

,Anno,Tipo produzione,Regione,Provincia,Fonte,Produzione (GWh)
0,2000,Lorda,Abruzzo,Chieti,Eolico,0.186600
71,2000,Netta,Abruzzo,Chieti,Eolico,0.186600
68,2000,Lorda,Abruzzo,Chieti,Fotovoltaico,1.182084
72,2000,Netta,Abruzzo,Chieti,Fotovoltaico,1.182084
69,2000,Lorda,Abruzzo,Chieti,Idrico,452.350336
73,2000,Netta,Abruzzo,Chieti,Idrico,452.350336
70,2000,Lorda,Abruzzo,Chieti,Termoelettrico,65.457184
74,2000,Netta,Abruzzo,Chieti,Termoelettrico,65.457184
75,2000,Lorda,Abruzzo,L'Aquila,Eolico,5.952885
78,2000,Netta,Abruzzo,L'Aquila,Eolico,5.952885


In [15]:
# Inspect gross and net hydropower production for Sassari, Sardinia.
df_2000_clean[
    (df_2000_clean["Regione"] == "Sardegna") &
    (df_2000_clean["Provincia"] == "Sassari") &
    (df_2000_clean["Fonte"] == "Idrico")
]

,Anno,Tipo produzione,Regione,Provincia,Fonte,Produzione (GWh)
5,2000,Netta,Sardegna,Sassari,Idrico,15.63208
8,2000,Lorda,Sardegna,Sassari,Idrico,15.63208


In [16]:
# Load Terna API credentials from the local environment file.



load_dotenv("../.env")

terna_client_id = os.getenv("TERNA_CLIENT_ID")
terna_client_secret = os.getenv("TERNA_CLIENT_SECRET")

print(terna_client_id is not None)
print(terna_client_secret is not None)

True
True


In [17]:
# Request an access token from the Terna API using client credentials.
token_url = "https://api.terna.it/public-api/access-token"

payload = {
    "client_id": terna_client_id,
    "client_secret": terna_client_secret,
    "grant_type": "client_credentials"
}

response = requests.post(token_url, data=payload)
response.raise_for_status()

print(response.status_code)

200


In [18]:
# Extract the access token from the API response.
token_data = response.json()

access_token = token_data["access_token"]

print(access_token is not None)

True


In [19]:
# Prepare the authorization header for authenticated API requests.
headers = {
    "Authorization": f"Bearer {access_token}"
}

In [20]:
# Request renewable electricity production data for 2000 from the Terna API.
api_url = "https://api.terna.it/generation/v2.0/renewable-sources-production"

params = {
    "year": 2000
}
time.sleep(1.2)
api_response = requests.get(
    api_url,
    headers=headers,
    params=params
)
api_response.raise_for_status()
print(api_response.status_code)

200


In [21]:
# Parse the API response and inspect its Python data type.
api_data = api_response.json()

type(api_data)

dict

In [22]:
api_data.keys()

dict_keys(['result', 'renewable_sources'])

In [23]:
type(api_data["renewable_sources"])

list

In [24]:
len(api_data["renewable_sources"])

1000

In [25]:
api_data["renewable_sources"][0]

{'year': 2000,
 'production_type': 'Netta',
 'region': 'Abruzzo',
 'province': 'Chieti',
 'renewable_source': 'Idrico',
 'production_GWh': 446.89218}

In [26]:
# Convert the API records into a pandas DataFrame.
df_api_2000 = pd.DataFrame(api_data["renewable_sources"])

df_api_2000.head()

,year,production_type,region,province,renewable_source,production_GWh
0,2000,Netta,Abruzzo,Chieti,Idrico,446.892180
1,2000,Lorda,Abruzzo,Chieti,Bioenergie,0.000000
2,2000,Netta,Abruzzo,Chieti,Bioenergie,0.000000
3,2000,Netta,Abruzzo,Chieti,Geotermoelettrico,0.000000
4,2000,Netta,Abruzzo,Chieti,Fotovoltaico,1.182084


In [27]:
api_data["result"]

{'message': 'Operation completed for the following input. year: 2000',
 'status': 'Completed'}

In [28]:
df_api_2000["renewable_source"].value_counts()

renewable_source
Idrico               200
Bioenergie           200
Geotermoelettrico    200
Fotovoltaico         200
Eolico               200
Name: count, dtype: int64

In [29]:
df_api_2000["production_type"].value_counts()

production_type
Netta    500
Lorda    500
Name: count, dtype: int64

In [30]:
# Count the number of distinct provinces returned by the API.
df_api_2000["province"].nunique()

100

In [31]:
# Check the number of records returned for each province.
df_api_2000.groupby("province").size().value_counts()

10    100
Name: count, dtype: int64

In [32]:
# Count the number of distinct regions returned by the API.
df_api_2000["region"].nunique()

20

In [33]:
# Inspect renewable production records for Chieti, Abruzzo.
df_api_2000[
    (df_api_2000["region"] == "Abruzzo") &
    (df_api_2000["province"] == "Chieti")
]

,year,production_type,region,province,renewable_source,production_GWh
0,2000,Netta,Abruzzo,Chieti,Idrico,446.892180
1,2000,Lorda,Abruzzo,Chieti,Bioenergie,0.000000
2,2000,Netta,Abruzzo,Chieti,Bioenergie,0.000000
3,2000,Netta,Abruzzo,Chieti,Geotermoelettrico,0.000000
4,2000,Netta,Abruzzo,Chieti,Fotovoltaico,1.182084
5,2000,Netta,Abruzzo,Chieti,Eolico,0.186600
6,2000,Lorda,Abruzzo,Chieti,Idrico,452.350340
7,2000,Lorda,Abruzzo,Chieti,Geotermoelettrico,0.000000
8,2000,Lorda,Abruzzo,Chieti,Fotovoltaico,1.182084
9,2000,Lorda,Abruzzo,Chieti,Eolico,0.186600


In [34]:
df_api_2000.columns

Index(['year', 'production_type', 'region', 'province', 'renewable_source',
       'production_GWh'],
      dtype='object')

In [35]:
# Check for missing values in the API response.
df_api_2000.isna().sum()

year                0
production_type     0
region              0
province            0
renewable_source    0
production_GWh      0
dtype: int64

In [36]:
# Check for duplicate records in the API response.
df_api_2000.duplicated().sum()

np.int64(0)

In [37]:
df_api_2000.dtypes

year                  int64
production_type      object
region               object
province             object
renewable_source     object
production_GWh      float64
dtype: object

In [38]:
# Calculate total gross renewable electricity production for 2000.
df_api_2000[
    df_api_2000["production_type"] == "Lorda"
]["production_GWh"].sum()

np.float64(50978.4734387)

In [39]:
# Calculate total net renewable electricity production for 2000.
df_api_2000[
    df_api_2000["production_type"] == "Netta"
]["production_GWh"].sum()

np.float64(50027.12902173)

In [40]:
# Calculate gross renewable electricity production by source for 2000.
df_api_2000[
    df_api_2000["production_type"] == "Lorda"
].groupby("renewable_source")["production_GWh"].sum()

renewable_source
Bioenergie            1504.469312
Eolico                 563.072870
Fotovoltaico             6.293099
Geotermoelettrico     4705.191600
Idrico               44199.446558
Name: production_GWh, dtype: float64

In [41]:
# Refresh the access token before the 2024 API request.
time.sleep(1.2)
response = requests.post(token_url, data=payload)
response.raise_for_status()

access_token = response.json()["access_token"]

headers = {
    "Authorization": f"Bearer {access_token}"
}

In [42]:
# Request renewable electricity production data for 2024 from the Terna API.
params = {
    "year": 2024
}
time.sleep(1.2)
api_response_2024 = requests.get(
    api_url,
    headers=headers,
    params=params
)
api_response_2024.raise_for_status()

print(api_response_2024.status_code)

200


In [43]:
# Parse the 2024 API response and inspect its structure.
api_data_2024 = api_response_2024.json()

api_data_2024.keys()

dict_keys(['result', 'renewable_sources'])

In [44]:
len(api_data_2024["renewable_sources"])

1070

In [45]:
# Convert the 2024 API records into a pandas DataFrame.
df_api_2024 = pd.DataFrame(api_data_2024["renewable_sources"])

In [46]:
df_api_2024["renewable_source"].value_counts()

renewable_source
Bioenergie           214
Geotermoelettrico    214
Fotovoltaico         214
Eolico               214
Idrico               214
Name: count, dtype: int64

In [47]:
# Count the number of distinct provinces returned by the API.
df_api_2024["province"].nunique()

107

In [48]:
# Count the number of distinct regions returned by the API.
df_api_2024["region"].nunique()

20

In [49]:
# Check for missing values in the 2024 API response.
df_api_2024.isna().sum()

year                0
production_type     0
region              0
province            0
renewable_source    0
production_GWh      0
dtype: int64

In [50]:
# Calculate total gross renewable electricity production for 2024.
df_api_2024[
    df_api_2024["production_type"] == "Lorda"
]["production_GWh"].sum()

np.float64(134357.50495734)

## Complete Time Series Collection

Renewable electricity production data are collected from the Terna API for each year from 2000 to 2024.

In [51]:
# Define the complete analysis period from 2000 to 2024.
years = list(range(2000, 2025))

years

[2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024]

In [52]:
# Initialize a list to collect API records for all years.
all_data = []

In [53]:
# Refresh the access token before downloading the complete time series.
time.sleep(1.2)
response = requests.post(token_url, data=payload)
response.raise_for_status()

print(response.status_code)

token_data = response.json()
access_token = token_data["access_token"]

headers = {
    "Authorization": f"Bearer {access_token}"
}

200


In [54]:
# Define the complete analysis period from 2000 to 2024.
years = list(range(2000, 2025))

# Initialize a list to collect API records for all years.
all_data = []

In [55]:
# Pause before the first API request after refreshing the access token.
time.sleep(1.2)


# Download renewable electricity production data for each year.
for year in years:
    params = {"year": year}

    response = requests.get(
        api_url,
        headers=headers,
        params=params
    )
    response.raise_for_status()

    print(year, response.status_code)

    year_data = response.json()["renewable_sources"]
    all_data.extend(year_data)

    time.sleep(1.2)

2000 200
2001 200
2002 200
2003 200
2004 200
2005 200
2006 200
2007 200
2008 200
2009 200
2010 200
2011 200
2012 200
2013 200
2014 200
2015 200
2016 200
2017 200
2018 200
2019 200
2020 200
2021 200
2022 200
2023 200
2024 200


## Data Quality Validation

The complete dataset is validated by checking temporal coverage, missing values, duplicate records, geographical coverage, renewable source coverage, and the presence of invalid production values.

In [56]:
# Convert all collected API records into a single DataFrame.
df = pd.DataFrame(all_data)

df.shape

(26540, 6)

In [57]:
df.head()

,year,production_type,region,province,renewable_source,production_GWh
0,2000,Netta,Abruzzo,Chieti,Idrico,446.892180
1,2000,Lorda,Abruzzo,Chieti,Bioenergie,0.000000
2,2000,Netta,Abruzzo,Chieti,Bioenergie,0.000000
3,2000,Netta,Abruzzo,Chieti,Geotermoelettrico,0.000000
4,2000,Netta,Abruzzo,Chieti,Fotovoltaico,1.182084


In [58]:
# Check that all 25 years are present in the dataset.
df["year"].nunique()

25

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26540 entries, 0 to 26539
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              26540 non-null  int64  
 1   production_type   26540 non-null  object 
 2   region            26540 non-null  object 
 3   province          26540 non-null  object 
 4   renewable_source  26540 non-null  object 
 5   production_GWh    26540 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 1.2+ MB


In [60]:
# Check for duplicate records in the complete dataset.
df.duplicated().sum()

np.int64(0)

In [61]:
# Check the number of records for each renewable energy source.
df["renewable_source"].value_counts()

renewable_source
Idrico               5308
Bioenergie           5308
Geotermoelettrico    5308
Fotovoltaico         5308
Eolico               5308
Name: count, dtype: int64

In [62]:
# Check the number of gross and net production records.
df["production_type"].value_counts()

production_type
Netta    13270
Lorda    13270
Name: count, dtype: int64

In [63]:
# Check the number of distinct regions available for each year.
df.groupby("year")["region"].nunique()

year
2000    20
2001    20
2002    20
2003    20
2004    20
2005    20
2006    20
2007    20
2008    20
2009    20
2010    20
2011    20
2012    20
2013    20
2014    20
2015    20
2016    20
2017    20
2018    20
2019    20
2020    20
2021    20
2022    20
2023    20
2024    20
Name: region, dtype: int64

In [64]:
# Check the number of distinct renewable energy sources available for each year.
df.groupby("year")["renewable_source"].nunique()

year
2000    5
2001    5
2002    5
2003    5
2004    5
2005    5
2006    5
2007    5
2008    5
2009    5
2010    5
2011    5
2012    5
2013    5
2014    5
2015    5
2016    5
2017    5
2018    5
2019    5
2020    5
2021    5
2022    5
2023    5
2024    5
Name: renewable_source, dtype: int64

In [65]:
# Check how the number of distinct provinces varies across years.
df.groupby("year")["province"].nunique()

year
2000    100
2001     99
2002    100
2003    100
2004    100
2005    100
2006    105
2007    107
2008    107
2009    110
2010    110
2011    110
2012    110
2013    110
2014    110
2015    110
2016    110
2017    107
2018    107
2019    107
2020    107
2021    107
2022    107
2023    107
2024    107
Name: province, dtype: int64

In [66]:
# Check for negative electricity production values.
(df["production_GWh"] < 0).sum()

np.int64(0)

In [67]:
# Count zero electricity production values.
(df["production_GWh"] == 0).sum()

np.int64(10280)

In [68]:
# Compare total gross and net electricity production.
df.groupby("production_type")["production_GWh"].sum()

production_type
Lorda    2.150180e+06
Netta    2.097986e+06
Name: production_GWh, dtype: float64

## Analytical Dataset

The Terna API provides both gross (`Lorda`) and net (`Netta`) electricity production. The analytical dataset uses gross production only, so the two measures are not combined or double-counted. After filtering, the `production_type` column is removed because it contains a single constant value.

In [69]:
# Keep only gross production to avoid double-counting gross and net values.
df_gross = df[df["production_type"] == "Lorda"].copy()

df_gross.shape

(13270, 6)

In [70]:
# Remove the production type column, which is now constant.
df_gross = df_gross.drop(columns=["production_type"])

df_gross.head()

,year,region,province,renewable_source,production_GWh
1,2000,Abruzzo,Chieti,Bioenergie,0.000000
6,2000,Abruzzo,Chieti,Idrico,452.350340
7,2000,Abruzzo,Chieti,Geotermoelettrico,0.000000
8,2000,Abruzzo,Chieti,Fotovoltaico,1.182084
9,2000,Abruzzo,Chieti,Eolico,0.186600


In [71]:
# Reset the index after filtering the dataset.
df_gross = df_gross.reset_index(drop=True)

df_gross.head()

,year,region,province,renewable_source,production_GWh
0,2000,Abruzzo,Chieti,Bioenergie,0.000000
1,2000,Abruzzo,Chieti,Idrico,452.350340
2,2000,Abruzzo,Chieti,Geotermoelettrico,0.000000
3,2000,Abruzzo,Chieti,Fotovoltaico,1.182084
4,2000,Abruzzo,Chieti,Eolico,0.186600


In [72]:
# Save the cleaned gross production dataset for subsequent analyses.
df_gross.to_csv(
    PROCESSED_DATA_DIR / "terna_renewable_production_2000_2024.csv",
    index=False
)